# 03 - Data Cleaning and Standardization
### Credit Portfolio Segmentation and Risk Pattern Analysis

---

## 📌 Beginner Objective
Raw data is rarely ready for modeling. In this notebook, we perform **Data Cleaning**:
1. **Rename columns** to standard, friendly names (`annual_income`, `loan_amount`, `dti`, `fico_score`).
2. **Handle missing values** by filling them with the **median** (the median is resistant to extreme outliers).
3. **Filter out extreme / corrupt outliers** (e.g., negative incomes or invalid DTI ratios).
4. **Save the clean dataset** to `../data/processed/cleaned_loans_dataset.csv`.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load raw data
raw_path = Path("../data/raw/loans_granting_model_dataset.csv")
if not raw_path.exists():
    raw_path = Path("data/raw/loans_granting_model_dataset.csv")

df = pd.read_csv(raw_path, nrows=25000)
print(f"Raw shape: {df.shape}")


---
## 1. Rename Columns to Standard Names
Standard names make our code readable and clear for anyone reviewing our work.


In [ ]:
# Mapping dictionary: {old_name: new_name}
column_mapping = {
    'revenue': 'annual_income',
    'loan_amnt': 'loan_amount',
    'dti_n': 'dti',
    'fico_n': 'fico_score',
    'experience_c': 'emp_length_years',
    'home_ownership_n': 'home_ownership'
}

df_clean = df.rename(columns=column_mapping)

# Keep relevant columns
columns_to_keep = [
    'annual_income', 'dti', 'loan_amount', 'fico_score', 
    'emp_length_years', 'home_ownership', 'purpose', 'addr_state', 'Default'
]
df_clean = df_clean[[col for col in columns_to_keep if col in df_clean.columns]].copy()
print("Selected columns:", df_clean.columns.tolist())


---
## 2. Handle Missing Values
Why use the median instead of the mean?
If 99 people earn $50,000 and 1 person earns $10,000,000, the mean is skewed to $150,000! 
The median remains $50,000, making it much safer for financial data.


In [ ]:
num_cols = ['annual_income', 'dti', 'loan_amount', 'fico_score', 'emp_length_years']

for col in num_cols:
    if col in df_clean.columns:
        median_val = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_val)
        print(f"Filled missing values in '{col}' with median: {median_val}")

# Verify no nulls remain in numeric columns
print("\nRemaining nulls:")
print(df_clean[num_cols].isnull().sum())


---
## 3. Remove Impossible / Corrupt Outliers
In credit data, we remove extreme records that would distort distance-based clustering:
- Annual income < $5,000 or > $1,000,000.
- Debt-to-income (DTI) < 0% or > 100%.


In [ ]:
initial_len = len(df_clean)

df_clean = df_clean[(df_clean['annual_income'] >= 5000) & (df_clean['annual_income'] <= 1000000)]
df_clean = df_clean[(df_clean['dti'] >= 0) & (df_clean['dti'] <= 100)]

print(f"Removed {initial_len - len(df_clean)} outlier rows. Clean dataset rows: {len(df_clean):,}")


---
## 4. Save Cleaned Dataset
We now save our pristine dataset into `../data/processed/cleaned_loans_dataset.csv`.


In [ ]:
output_path = Path("../data/processed/cleaned_loans_dataset.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(output_path, index=False)
print(f"Successfully saved clean dataset to: {output_path.resolve()}")
df_clean.head(3)
